## 🛠️ Importing Required Libraries

In [ ]:
from tqdm import tqdm
import numpy as np
import pandas as pd
from itertools import accumulate
import matplotlib.pyplot as plt
from torchtext.data.utils import get_tokenizer

import torch
import torch.nn as nn

from torch.utils.data import DataLoader
import numpy as np
from torchtext.datasets import AG_NEWS
from IPython.display import Markdown as md
from tqdm import tqdm

from torchtext.vocab import build_vocab_from_iterator
from torchtext.datasets import AG_NEWS
from torch.utils.data.dataset import random_split
from torchtext.data.functional import to_map_style_dataset
from sklearn.manifold import TSNE
import plotly.graph_objs as go
from sklearn.model_selection import train_test_split

from torchtext.data.utils import get_tokenizer

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

## ⚙️ Helper Functions

Defining the plotting and evaluation functions.

In [ ]:
def plot(COST,ACC):
    fig, ax1 = plt.subplots()
    color = 'tab:red'
    ax1.plot(COST, color=color)
    ax1.set_xlabel('epoch', color=color)
    ax1.set_ylabel('total loss', color=color)
    ax1.tick_params(axis='y', color=color)
    
    ax2 = ax1.twinx()  
    color = 'tab:blue'
    ax2.set_ylabel('accuracy', color=color)  # you already handled the x-label with ax1
    ax2.plot(ACC, color=color)
    ax2.tick_params(axis='y', color=color)
    fig.tight_layout()  # otherwise the right y-label is slightly clipped
    
    plt.show()

## 📥 Data Loading and Preprocessing

Load the **AG_NEWS** dataset, find the number of classes, and build the vocabulary.

In [ ]:
# Load the training data iterator
train_iter= iter(AG_NEWS(split="train"))

# Sample a data point to see the format (label, text)
y,text= next((train_iter))
print(f"Sample label: {y}, Sample text: {text}")

# Define class labels and find the total number of classes
ag_news_label = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tec"}
# Reinitialize to count all classes
train_iter = AG_NEWS(split="train")
num_class = len(set([label for (label, text) in train_iter ]))
print(f"Number of classes: {num_class}")

### Vocabulary Creation

Define the tokenizer and build the vocabulary from the training data.

In [ ]:
# Reinitialize train_iter
train_iter = AG_NEWS(split="train")

# Define tokenizer and yield_tokens function for vocabulary building
tokenizer = get_tokenizer("basic_english")

def yield_tokens(data_iter):
    for _, text in data_iter:
        yield tokenizer(text.lower())  # Lowercase conversion for consistency

# Build vocabulary
vocab = build_vocab_from_iterator(yield_tokens(train_iter), specials=["<unk>"])
vocab.set_default_index(vocab["<unk>"])
vocab_size=len(vocab)

print(f"Vocabulary size: {vocab_size}")
print(f"Token indices for 'age' and 'hello': {vocab(['age', 'hello'])}")

### Dataset Splitting

Convert iterable datasets to map-style and split the training data into training and validation sets.

In [ ]:
# Split the dataset into training and testing iterators.
train_iter, test_iter = AG_NEWS()

# Convert the training and testing iterators to map-style datasets.
train_dataset = to_map_style_dataset(train_iter)
test_dataset = to_map_style_dataset(test_iter)

# Determine the number of samples for training and validation (95% train, 5% validation).
num_train = int(len(train_dataset) * 0.95)

# Randomly split the training dataset
split_train_, split_valid_ = random_split(train_dataset, [num_train, len(train_dataset) - num_train])

# Check for CUDA device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 📦 Data Loader

Define the text/label pipelines and the `collate_batch` function for batching data, then create the data loaders.

In [ ]:
def text_pipeline(x):
  """Tokenizes text and converts tokens to vocabulary indices."""
  return vocab(tokenizer(x))

def label_pipeline(x):
   """Converts raw label (1-based) to 0-based index."""
   return int(x) - 1

def collate_batch(batch):
    """Custom collate function to process a batch of data."""
    label_list, text_list, offsets = [], [], [0]
    for _label, _text in batch:
        label_list.append(label_pipeline(_label))
        processed_text = torch.tensor(text_pipeline(_text), dtype=torch.int64)
        text_list.append(processed_text)
        offsets.append(processed_text.size(0))
    label_list = torch.tensor(label_list, dtype=torch.int64)
    offsets = torch.tensor(offsets[:-1]).cumsum(dim=0)
    text_list = torch.cat(text_list)
    return label_list.to(device), text_list.to(device), offsets.to(device)

In [ ]:
BATCH_SIZE = 64

train_dataloader = DataLoader(
    split_train_, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch
)
valid_dataloader = DataLoader(
    split_valid_, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch
)
test_dataloader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch
)

# Verify data loader output shape
label, text, offsets=next(iter(valid_dataloader ))
print(f"Sample label tensor: {label.shape}, Sample text tensor: {text.shape}, Sample offsets tensor: {offsets.shape}")

## 🧠 Neural Network Model

Define the **`TextClassificationModel`** using **`nn.EmbeddingBag`** and a fully connected layer.

In [ ]:
from torch import nn

class TextClassificationModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super(TextClassificationModel, self).__init__()
        self.embedding = nn.EmbeddingBag(vocab_size, embed_dim, sparse=False)
        self.fc = nn.Linear(embed_dim, num_class)
        self.init_weights()

    def init_weights(self):
        initrange = 0.5
        self.embedding.weight.data.uniform_(-initrange, initrange)
        self.fc.weight.data.uniform_(-initrange, initrange)
        self.fc.bias.data.zero_()

    def forward(self, text, offsets):
        embedded = self.embedding(text, offsets)
        return self.fc(embedded)

# Initialize model parameters
emsize=64

# Create the model instance
model = TextClassificationModel(vocab_size, emsize, num_class).to(device)
print(model)

# Verify output shape
predicted_label=model(text, offsets)
print(f"Model output shape: {predicted_label.shape}")

### Prediction and Evaluation Functions

Define functions for predicting a single article's class and evaluating the model's accuracy on a dataset.

In [ ]:
def predict(text, text_pipeline):
    model.eval() # Set model to evaluation mode
    with torch.no_grad():
        text = torch.tensor(text_pipeline(text)).to(device)
        # offsets is [0] because it's a single text sequence
        output = model(text, torch.tensor([0]).to(device))
        # argmax to get the predicted class index, +1 to convert back to 1-based label, then map to string
        return ag_news_label[output.argmax(1).item() + 1]

def evaluate(dataloader):
    model.eval()
    total_acc, total_count= 0, 0

    with torch.no_grad():
        for idx, (label, text, offsets) in enumerate(dataloader):
            predicted_label = model(text, offsets)

            total_acc += (predicted_label.argmax(1) == label).sum().item()
            total_count += label.size(0)
    return total_acc / total_count

# Evaluate untrained model
print(f"Accuracy on test data (untrained model): {evaluate(test_dataloader):.4f}")

## 🏋️ Model Training

In [ ]:
# Training setup
LR=0.1

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, 1.0, gamma=0.1)
print(scheduler)

In [ ]:
EPOCHS = 10
cum_loss_list=[]
acc_epoch=[]
acc_old=0

for epoch in tqdm(range(1, EPOCHS + 1)):
    model.train()
    cum_loss=0
    for idx, (label, text, offsets) in enumerate(train_dataloader):
        optimizer.zero_grad()
        predicted_label = model(text, offsets)
        loss = criterion(predicted_label, label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
        optimizer.step()
        cum_loss+=loss.item()

    cum_loss_list.append(cum_loss)
    accu_val = evaluate(valid_dataloader)
    acc_epoch.append(accu_val)

    # Save model if validation accuracy improves
    if accu_val > acc_old:
      acc_old= accu_val
      torch.save(model.state_dict(), 'my_model.pth')
    
    scheduler.step() # Step the scheduler after the epoch

### Results Visualization

In [ ]:
plot(cum_loss_list,acc_epoch)

### Final Test Evaluation

In [ ]:
print(f"Accuracy on test data (trained model): {evaluate(test_dataloader):.4f}")

## 🎯 Single Article Prediction

In [ ]:
article="""Canada navigated a stiff test against the Republic of Ireland on a rain soaked evening in Perth, coming from behind to claim a vital 2-1 victory at the Women’s World Cup.\nKatie McCabe opened the scoring with an incredible Olimpico goal – scoring straight from a corner kick – as her corner flew straight over the despairing Canada goalkeeper Kailen Sheridan at Perth Rectangular Stadium in Australia.\nJust when Ireland thought it had safely navigated itself to half time with a lead, Megan Connolly failed to get a clean connection on a clearance with the resulting contact squirming into her own net to level the score.\nMinutes into the second half, Adriana Leon completed the turnaround for the Olympic champion, slotting home from the edge of the area to seal the three points."""

In [ ]:
result = predict(article, text_pipeline)

markdown_content = f'''
<div style="background-color: lightgray; padding: 10px;">
    <h3>{article}</h3>
    <h4>The category of the news article: {result}</h4>
</div>
'''

md(markdown_content)

---
## 🧑‍💻 Exercises

### Exercise 1 - Load the pre-trained model (path = 'my_model.pth').

In [ ]:
model.load_state_dict(torch.load('my_model.pth'))
model.eval()

### Exercise 2 - Define the list of new articles for classification.

In [ ]:
new_articles = [
    "International talks have made significant headway with the signing of a climate accord that commits countries to reduce emissions by 40% over the next two decades. World leaders expressed optimism at the conclusion of the summit.",
    "In a stunning upset, the underdog team won the national title, beating the favorites in a match that featured an incredible comeback and a last-minute goal that sealed their victory in front of a record crowd.",
    "Market analysts are optimistic as the tech startup's stock prices soared after the announcement of their latest product, which promises to revolutionize how we interact with smart devices.",
    "A recent study published in a leading scientific journal suggests that a new drug has shown promise in the treatment of Alzheimer's disease, outperforming current leading medications in early clinical trials.",
    "Diplomatic relations have taken a positive turn with the recent peace talks that aim to end decades of conflict. The ceasefire agreement has been welcomed by the international community.",
    "Economic indicators show a sharp rebound in manufacturing, with the automobile industry leading the charge. Analysts predict this surge will result in significant job creation over the next year.",
    "Researchers at the university's astrophysics department have discovered a potentially habitable exoplanet. The planet, which lies in a nearby star system, has conditions that could support liquid water and, possibly, life.",
    "The sports world is in shock as a legendary player announces their retirement. Over an illustrious 20-year career, the athlete has amassed numerous records and is regarded as one of the greatest to ever play the game.",
    "A multinational corporation has announced a major investment in renewable energy. The initiative includes the construction of new wind farms and solar panels that will power hundreds of thousands of homes.",
    "Climate scientists warn that the melting of the polar ice caps has been accelerating at an alarming rate, raising sea levels and threatening coastal cities worldwide with increased flooding risks."
]

### Exercise 3 - Classify each article and display the results.

In [ ]:
for i, article in enumerate(new_articles, start=1):
    prediction = predict(article, text_pipeline)
    print(f"Article {i} is classified as: {prediction}\n")

---
## 🎉 Completed Text Classification